# Agentic RAG Challenge -- Student Academic Intelligence System

---

## Objective

Build an **Agentic RAG pipeline** that answers 50 test queries about student performance by reasoning over structured data (CSVs) and unstructured policy documents (Markdown).

## Submission Format

| Column | Type | Description |
|--------|------|-------------|
| `answerable` | 0 or 1 | 1 = answerable, 0 = out-of-scope |
| `lookup_value` | float | Raw data value |
| `rule_value` | float | Policy rule applied (0 if no rule needed) |
| `final_answer` | float | Computed result |

**Out-of-scope:** `answerable=0, lookup_value=-1, rule_value=-1, final_answer=-1`

**Evaluation:** RMSE across all 4 columns. Lower is better.

---

> Follow the cells sequentially. Complete sections marked TODO.
>
> **IMPORTANT:** The submission generation cell (Cell 11) MUST be executed for your notebook to produce a valid submission on Kaggle.
>
> **Rate limits:** The notebook ships with a `KeyPool` that spreads calls across your Groq keys and cools down any key that hits a 429. Add **at least 3-4 Groq keys** as Kaggle Secrets -- 50 queries then finish in roughly 10 minutes. Progress is saved to `submission_partial.csv` every 10 queries.

## Cell 1 -- Install Dependencies

> Run this cell first. After installation, restart the kernel and run all cells from the top.

In [ ]:
print("Installing required packages...\n")

!pip install -q \
    openai \
    google-genai \
    faiss-cpu \
    pandas \
    numpy

print("\nInstallation complete.")
print("Please restart the kernel/runtime before running the next cell.")

## Cell 2 -- Configuration

Set your data path and model preferences here. All remaining cells use this configuration.

In [7]:

# ============================================
# CELL 2: CONFIGURATION
# ============================================

# Main Kaggle Dataset Path
DATA_PATH = "/kaggle/input/competitions/gen-ai-project-agentic-rag/Gen AI Project 3"

# Individual Folder Paths
DATASET_PATH = f"{DATA_PATH}/Dataset"
QUERY_PATH = f"{DATA_PATH}/Query Files"
POLICY_PATH = f"{DATA_PATH}/Policy Documents"

# LLM and Embedding Models
LLM_MODEL = "openai/gpt-oss-120b"
EMBED_MODEL = "gemini-embedding-001"

# Display Configuration
print("Configuration Loaded Successfully!")
print("------------------------------------")
print(f"Data path: {DATA_PATH}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Query path: {QUERY_PATH}")
print(f"Policy path: {POLICY_PATH}")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")


Configuration Loaded Successfully!
------------------------------------
Data path: /kaggle/input/competitions/gen-ai-project-agentic-rag/Gen AI Project 3
Dataset path: /kaggle/input/competitions/gen-ai-project-agentic-rag/Gen AI Project 3/Dataset
Query path: /kaggle/input/competitions/gen-ai-project-agentic-rag/Gen AI Project 3/Query Files
Policy path: /kaggle/input/competitions/gen-ai-project-agentic-rag/Gen AI Project 3/Policy Documents
LLM: openai/gpt-oss-120b
Embeddings: gemini-embedding-001


## Cell 3 -- Imports

> Run this cell without modification.

In [2]:
import os, json, re, time, csv, math
import numpy as np
import pandas as pd
from datetime import date
from collections import defaultdict
import faiss
from google import genai
from openai import OpenAI

print("Imports loaded successfully.")

Imports loaded successfully.


## Cell 4 -- API Keys & Rate Limiting

Add your API keys as **Kaggle Secrets** before running this cell (Notebook menu > **Add-ons > Secrets**):
- `GROQ_API_KEY_1`, `GROQ_API_KEY_2`, ... up to `_8` (**use at least 3-4 keys**)
- `GOOGLE_API_KEY`

**Important:** after adding each secret, make sure its toggle is **ON (Attached)** for this notebook, and that **Internet** is enabled in the notebook settings.

`KeyPool` picks a key that is under its per-minute limit and puts any key that returns a 429 on cooldown.


In [3]:
# ---- Load API keys from Kaggle Secrets (Add-ons > Secrets) ----
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

def _secret(name):
    try:
        v = secrets.get_secret(name)
        return v.strip() if v else None
    except Exception:
        return None  # secret not added / not attached to this notebook

# Groq keys: GROQ_API_KEY_1 ... GROQ_API_KEY_8 (gaps are skipped, duplicates removed)
GROQ_KEYS = []
for i in range(1, 9):
    k = _secret(f"GROQ_API_KEY_{i}")
    if k and k not in GROQ_KEYS:
        GROQ_KEYS.append(k)

GOOGLE_API_KEY = _secret("GOOGLE_API_KEY")

if not GROQ_KEYS:
    raise RuntimeError(
        "No Groq keys found. In Kaggle: Add-ons > Secrets, add GROQ_API_KEY_1, GROQ_API_KEY_2, ... "
        "and toggle each one ON ('Attached') for this notebook."
    )
if not GOOGLE_API_KEY:
    raise RuntimeError("GOOGLE_API_KEY secret missing or not attached to this notebook.")

print(f"Kaggle secrets loaded -- {len(GROQ_KEYS)} Groq key(s) + Google key.")
if len(GROQ_KEYS) < 3:
    print("WARNING: fewer than 3 Groq keys. Expect slow runs / rate-limit waits.")

# Gemini client (embeddings)
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)


class KeyPool:
    """Round-robin Groq keys with a per-key requests-per-minute cap and 429 cooldown."""
    def __init__(self, keys, rpm_per_key=20):
        self.keys = keys
        self.rpm = rpm_per_key
        self.calls = {k: [] for k in keys}
        self.cooldown_until = {k: 0.0 for k in keys}
        self.idx = 0

    def get(self):
        """Return a key that is under its RPM cap and not cooling down (waits if all are busy)."""
        while True:
            now = time.time()
            for _ in range(len(self.keys)):
                k = self.keys[self.idx]
                self.idx = (self.idx + 1) % len(self.keys)
                self.calls[k] = [t for t in self.calls[k] if now - t < 60]
                if now >= self.cooldown_until[k] and len(self.calls[k]) < self.rpm:
                    self.calls[k].append(now)
                    return k
            time.sleep(1)  # every key busy -> wait

    def penalize(self, key, seconds=30):
        """Put a key on cooldown after a 429."""
        self.cooldown_until[key] = time.time() + seconds


# Tune rpm_per_key to your Groq limits (check console.groq.com/settings/limits)
pool = KeyPool(GROQ_KEYS, rpm_per_key=20)

print("Clients initialized.")


Kaggle secrets loaded -- 3 Groq key(s) + Google key.
Clients initialized.


## Cell 5 -- Load Data

> This cell is pre-filled -- run it to load the dataset.

In [8]:

# ============================================
# CELL 5: LOAD DATA
# ============================================

# Load structured data
courses_df = pd.read_csv(f"{DATASET_PATH}/courses.csv")
mentors_df = pd.read_csv(f"{DATASET_PATH}/mentors.csv")
students_df = pd.read_csv(f"{DATASET_PATH}/students.csv")
assessments_df = pd.read_csv(f"{DATASET_PATH}/assessments.csv")
attendance_df = pd.read_csv(f"{DATASET_PATH}/attendance.csv")

# Load test queries
test_df = pd.read_csv(f"{QUERY_PATH}/test.csv")

# Load training data (for validation)
train_df = pd.read_csv(f"{QUERY_PATH}/train.csv")

# Load policy documents
policy_docs = {}

policy_dir = POLICY_PATH

for fname in os.listdir(policy_dir):
    if fname.endswith(".md"):
        with open(os.path.join(policy_dir, fname), encoding="utf-8") as f:
            policy_docs[fname] = f.read()

# Display dataset information
print("Data Loaded Successfully!")
print("--------------------------------")

print(f"Courses: {len(courses_df)}, Mentors: {len(mentors_df)}, Students: {len(students_df)}")
print(f"Assessments: {len(assessments_df)}, Attendance: {len(attendance_df)}")

print(f"Test queries: {len(test_df)}, Train queries: {len(train_df)}")

print(f"Policy documents: {list(policy_docs.keys())}")


Data Loaded Successfully!
--------------------------------
Courses: 20, Mentors: 30, Students: 800
Assessments: 14072, Attendance: 18000
Test queries: 50, Train queries: 100
Policy documents: ['attendance_policy.md', 'faq.md', 'progress_milestones.md', 'grading_policy.md', 'alert_rules.md']


In [5]:
import os

print("Available Kaggle input folders:")

print(os.listdir("/kaggle/input"))

Available Kaggle input folders:
['competitions']


In [6]:
for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    if files:
        print("Files:", files[:10])

/kaggle/input
/kaggle/input/competitions
/kaggle/input/competitions/gen-ai-project-agentic-rag
/kaggle/input/competitions/gen-ai-project-agentic-rag/Gen AI Project 3
Files: ['starter_notebook.ipynb']
/kaggle/input/competitions/gen-ai-project-agentic-rag/Gen AI Project 3/Query Files
Files: ['sample_submission.csv', 'train.csv', 'test.csv']
/kaggle/input/competitions/gen-ai-project-agentic-rag/Gen AI Project 3/Dataset
Files: ['metadata.md', 'mentors.csv', 'attendance.csv', 'assessments.csv', 'courses.csv', 'students.csv']
/kaggle/input/competitions/gen-ai-project-agentic-rag/Gen AI Project 3/Policy Documents
Files: ['attendance_policy.md', 'faq.md', 'progress_milestones.md', 'grading_policy.md', 'alert_rules.md']


## Cell 6 -- Build Vector Store

### Your Task

Build a FAISS vector store from the policy documents for retrieval.

In [9]:

# STEP 1: Import embedding configuration
from google.genai import types

# STEP 2: Chunk policy documents
policy_chunks = []

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200

for filename, content in policy_docs.items():
    content = content.strip()
    start = 0

    while start < len(content):
        end = start + CHUNK_SIZE
        chunk_text = content[start:end].strip()

        if chunk_text:
            policy_chunks.append({
                "source": filename,
                "text": chunk_text
            })

        start += CHUNK_SIZE - CHUNK_OVERLAP

print(f"Created {len(policy_chunks)} policy chunks")


# STEP 3: Generate embeddings using Gemini
all_embeddings = []

for i in range(0, len(policy_chunks), 16):
    batch = policy_chunks[i:i + 16]

    response = gemini_client.models.embed_content(
        model=EMBED_MODEL,
        contents=[chunk["text"] for chunk in batch],
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_DOCUMENT"
        )
    )

    batch_embeddings = [
        embedding.values for embedding in response.embeddings
    ]

    all_embeddings.extend(batch_embeddings)

print("Policy embeddings generated successfully!")


# STEP 4: Build FAISS index
embedding_matrix = np.array(
    all_embeddings, dtype=np.float32
)

faiss.normalize_L2(embedding_matrix)

embedding_dimension = embedding_matrix.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)
faiss_index.add(embedding_matrix)

print(f"FAISS index created with {faiss_index.ntotal} vectors")


# STEP 5: Implement policy retrieval
def retrieve_policies(query, top_k=3):
    """Retrieve the top-k most relevant policy chunks."""

    response = gemini_client.models.embed_content(
        model=EMBED_MODEL,
        contents=query,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY"
        )
    )

    query_embedding = np.array(
        [response.embeddings[0].values],
        dtype=np.float32
    )

    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(
        query_embedding,
        min(top_k, len(policy_chunks))
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            results.append({
                "source": policy_chunks[idx]["source"],
                "text": policy_chunks[idx]["text"],
                "score": float(score)
            })

    return results


print("Policy retrieval function ready!")


Created 15 policy chunks
Policy embeddings generated successfully!
FAISS index created with 15 vectors
Policy retrieval function ready!


## Cell 7 -- LLM Helper

> This cell is pre-filled -- modify parameters if needed.

In [10]:
def call_llm(messages, tools=None, temperature=0.0, max_retries=6):
    """Call Groq LLM. Picks a key from the pool; only rate-limit errors trigger a key cooldown."""
    for attempt in range(max_retries):
        key = pool.get()
        client = OpenAI(api_key=key, base_url="https://api.groq.com/openai/v1")
        try:
            kwargs = {
                "model": LLM_MODEL,
                "messages": messages,
                "temperature": temperature,
            }
            if tools:
                kwargs["tools"] = tools
                kwargs["tool_choice"] = "auto"

            return client.chat.completions.create(**kwargs)
        except Exception as e:
            msg = str(e).lower()
            if "429" in msg or "rate" in msg:
                pool.penalize(key, 30)
            wait = min(2 ** attempt, 30)
            print(f"  [retry {attempt+1}/{max_retries}] {str(e)[:120]} -- waiting {wait}s...")
            time.sleep(wait)

    raise RuntimeError("LLM failed after all retries")

print("LLM helper ready.")

LLM helper ready.


In [11]:

print("COURSES:", courses_df.columns.tolist())
print("MENTORS:", mentors_df.columns.tolist())
print("STUDENTS:", students_df.columns.tolist())
print("ASSESSMENTS:", assessments_df.columns.tolist())
print("ATTENDANCE:", attendance_df.columns.tolist())

print("\n--- POLICY DOCUMENTS ---")
for filename, content in policy_docs.items():
    print(f"\n### {filename}")
    print(content)


COURSES: ['course_id', 'course_name', 'total_modules', 'total_sessions', 'passing_grade']
MENTORS: ['mentor_id', 'mentor_name', 'department']
STUDENTS: ['student_id', 'student_name', 'batch', 'course_enrolled', 'enrollment_date', 'current_module', 'mentor_id']
ASSESSMENTS: ['assessment_id', 'student_id', 'course_id', 'module', 'type', 'max_score', 'scored', 'due_date', 'submitted_date']
ATTENDANCE: ['student_id', 'date', 'session_type', 'status']

--- POLICY DOCUMENTS ---

### attendance_policy.md
# SkillBridge EdTech — Attendance Policy


## 1. Attendance Tracking

Attendance is recorded for every scheduled session. Each record includes:
- **student_id**: The student
- **date**: Session date
- **session_type**: One of `lecture`, `lab`, or `mentoring`
- **status**: One of `present`, `absent`, or `late`

## 2. Attendance Percentage Formula

The attendance percentage is calculated as:

```
attendance_pct = ((present_count + 0.5 × late_count) / total_sessions) × 100
```

Where:
- `present

## Cell 8 -- Define Agent Tools

### Your Task

Define the tool functions your agent will call and their OpenAI-format schemas.

Refer to the policy documents and dataset to determine what tools are needed and how they should compute results.

In [12]:

# CELL 8: Define Agent Tools

REFERENCE_DATE = pd.Timestamp("2025-01-15")


def find_student(student_name):
    """Find a student using their name."""
    matches = students_df[
        students_df["student_name"].astype(str).str.contains(
            str(student_name), case=False, na=False
        )
    ]

    if len(matches) == 0:
        return json.dumps({"error": "Student not found."})

    return matches.to_json(orient="records", date_format="iso")


def student_academic_summary(student_id):
    """Calculate attendance, grade, progress and alert for a student."""

    student_rows = students_df[
        students_df["student_id"].astype(str) == str(student_id)
    ]

    if student_rows.empty:
        return json.dumps({"error": "Student not found."})

    student = student_rows.iloc[0]
    sid = str(student["student_id"])

    # 1. Attendance calculation
    att = attendance_df[
        attendance_df["student_id"].astype(str) == sid
    ].copy()

    if len(att) > 0:
        weights = att["session_type"].astype(str).str.lower().map(
            lambda x: 1.5 if x == "lab" else 1.0
        )

        credits = att["status"].astype(str).str.lower().map(
            lambda x: 1.0 if x == "present"
            else 0.5 if x == "late" else 0.0
        )

        weighted_total = weights.sum()
        weighted_present = (weights * credits).sum()

        weighted_attendance = (
            weighted_present / weighted_total * 100
            if weighted_total > 0 else 0
        )

        total_sessions = len(att)
        raw_attendance = (
            (att["status"].astype(str).str.lower() == "present").sum()
            + 0.5 * (att["status"].astype(str).str.lower() == "late").sum()
        ) / total_sessions * 100
    else:
        weighted_attendance = 0
        raw_attendance = 0

    # 2. Grade calculation
    assessments = assessments_df[
        assessments_df["student_id"].astype(str) == sid
    ].copy()

    weights = {
        "quiz": 0.20,
        "assignment": 0.30,
        "project": 0.25,
        "exam": 0.25
    }

    component_averages = {}

    for component, weight in weights.items():
        type_values = assessments["type"].astype(str).str.lower().str.strip()

        if component == "quiz":
            selected = assessments[type_values.str.contains("quiz", na=False)]
        else:
            selected = assessments[type_values == component]

        total_max = pd.to_numeric(
            selected["max_score"], errors="coerce"
        ).sum()

        total_scored = pd.to_numeric(
            selected["scored"], errors="coerce"
        ).sum()

        component_averages[component] = (
            total_scored / total_max * 100 if total_max > 0 else 0
        )

    grade = sum(
        component_averages[c] * w for c, w in weights.items()
    )

    # 3. Progress calculation
    course_rows = courses_df[
        courses_df["course_id"].astype(str)
        == str(student["course_enrolled"])
    ]

    total_modules = (
        float(course_rows.iloc[0]["total_modules"])
        if not course_rows.empty else 0
    )

    enrollment_date = pd.to_datetime(student["enrollment_date"])
    weeks_since_enrollment = (
        REFERENCE_DATE - enrollment_date
    ).days / 7

    expected_module = min(
        weeks_since_enrollment / 3, total_modules
    ) if total_modules > 0 else 0

    current_module = float(student["current_module"])

    if current_module >= total_modules and total_modules > 0:
        progress_status = "Completed"
    elif current_module >= expected_module:
        progress_status = "On Track"
    elif current_module >= expected_module - 1:
        progress_status = "Slightly Behind"
    elif current_module >= expected_module - 3:
        progress_status = "Behind"
    else:
        progress_status = "At Risk"

    completion_pct = (
        current_module / total_modules * 100
        if total_modules > 0 else 0
    )

    # 4. Alert classification
    if grade < 50 or weighted_attendance < 60:
        alert = "Red"
    elif grade < 65 or weighted_attendance < 70:
        alert = "Yellow"
    elif grade >= 75 and weighted_attendance >= 75:
        alert = "Green"
    else:
        alert = "Orange"

    result = {
        "student_id": sid,
        "student_name": str(student["student_name"]),
        "batch": str(student["batch"]),
        "course": str(student["course_enrolled"]),
        "current_module": current_module,
        "expected_module": round(expected_module, 2),
        "progress_status": progress_status,
        "completion_pct": round(completion_pct, 2),
        "component_averages": {
            k: round(v, 2) for k, v in component_averages.items()
        },
        "overall_grade": round(grade, 2),
        "raw_attendance_pct": round(raw_attendance, 2),
        "weighted_attendance_pct": round(weighted_attendance, 2),
        "alert": alert
    }

    return json.dumps(result)


def batch_summary(batch_name):
    """Calculate average course completion for a batch."""

    batch_students = students_df[
        students_df["batch"].astype(str).str.lower()
        == str(batch_name).lower()
    ].copy()

    if batch_students.empty:
        return json.dumps({"error": "Batch not found."})

    merged = batch_students.merge(
        courses_df,
        left_on="course_enrolled",
        right_on="course_id",
        how="left"
    )

    merged["completion_pct"] = (
        pd.to_numeric(merged["current_module"], errors="coerce")
        / pd.to_numeric(merged["total_modules"], errors="coerce")
    ) * 100

    return json.dumps({
        "batch": batch_name,
        "total_students": int(len(merged)),
        "average_completion_pct": round(
            merged["completion_pct"].mean(), 2
        )
    })


def course_summary(course_id):
    """Calculate average completion for a course."""

    course_students = students_df[
        students_df["course_enrolled"].astype(str)
        == str(course_id)
    ].copy()

    if course_students.empty:
        return json.dumps({"error": "Course not found or no students enrolled."})

    course_rows = courses_df[
        courses_df["course_id"].astype(str) == str(course_id)
    ]

    if course_rows.empty:
        return json.dumps({"error": "Course details not found."})

    total_modules = float(course_rows.iloc[0]["total_modules"])

    average_completion = (
        pd.to_numeric(course_students["current_module"], errors="coerce")
        / total_modules * 100
    ).mean()

    return json.dumps({
        "course_id": str(course_id),
        "course_name": str(course_rows.iloc[0]["course_name"]),
        "total_students": int(len(course_students)),
        "average_completion_pct": round(average_completion, 2)
    })


# Map tool names to functions
TOOL_FUNCTIONS = {
    "find_student": find_student,
    "student_academic_summary": student_academic_summary,
    "batch_summary": batch_summary,
    "course_summary": course_summary
}


# OpenAI-format tool schemas
openai_tools = [
    {
        "type": "function",
        "function": {
            "name": "find_student",
            "description": "Find a student using their name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "student_name": {
                        "type": "string",
                        "description": "Student's name"
                    }
                },
                "required": ["student_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "student_academic_summary",
            "description": "Get a student's grade, attendance, progress and alert.",
            "parameters": {
                "type": "object",
                "properties": {
                    "student_id": {
                        "type": "string",
                        "description": "Unique student ID"
                    }
                },
                "required": ["student_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "batch_summary",
            "description": "Calculate average completion percentage for a batch.",
            "parameters": {
                "type": "object",
                "properties": {
                    "batch_name": {
                        "type": "string",
                        "description": "Batch name"
                    }
                },
                "required": ["batch_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "course_summary",
            "description": "Calculate average completion percentage for a course.",
            "parameters": {
                "type": "object",
                "properties": {
                    "course_id": {
                        "type": "string",
                        "description": "Course ID"
                    }
                },
                "required": ["course_id"]
            }
        }
    }
]

print(f"{len(TOOL_FUNCTIONS)} tools defined, {len(openai_tools)} schemas registered.")


4 tools defined, 4 schemas registered.


## Cell 9 -- Build Agent Pipeline

### Your Task

Implement the agent that takes a query, retrieves relevant policies, calls tools as needed, and returns the answer as a 4-column JSON.

In [13]:

# CELL 9: Build Agent Pipeline

SYSTEM_PROMPT = """
You are an intelligent Agentic RAG assistant for SkillBridge EdTech.

Your task is to answer questions using:
1. Structured student, course, assessment and attendance datasets.
2. Retrieved institutional policy documents.
3. The tools available to you.

IMPORTANT INSTRUCTIONS:

- Always use the available tools when structured dataset information is required.
- If a student name is provided, first use find_student to identify the student_id.
- Then use student_academic_summary to calculate the student's academic details.
- Use batch_summary for batch-level completion questions.
- Use course_summary for course-level completion questions.
- Follow the retrieved policy context for all policy-based calculations.
- Never invent student records, scores, attendance or policy rules.
- Do not use outside assumptions.
- The reference date for calculations is January 15, 2025.
- Attendance calculations must follow the attendance policy.
- Grade calculations must follow the grading policy.
- Progress calculations must follow the progress milestones policy.
- Alert calculations must follow the alert rules policy.

FINAL OUTPUT FORMAT:

Return ONLY one valid JSON object with these four keys:

{
  "answerable": 1,
  "lookup_value": 0.0,
  "rule_value": 0.0,
  "final_answer": 0.0
}

FIELD MEANINGS:

- answerable: 1 if the question can be answered using the provided data and policies; otherwise 0.
- lookup_value: Numeric value retrieved from structured data. Use -1 if not applicable.
- rule_value: Numeric value calculated using institutional policies. Use -1 if not applicable.
- final_answer: The final numeric answer requested by the question.

If the question is outside the scope of SkillBridge EdTech or cannot be answered from the available information, return:

{
  "answerable": 0,
  "lookup_value": -1,
  "rule_value": -1,
  "final_answer": -1
}

Do not include explanations, markdown, or text outside the JSON object.
"""


def run_agent(query, policy_context, max_turns=6):
    """Run the tool-calling agent. Returns the final response text."""

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
Question:
{query}

Retrieved Policy Context:
{policy_context}

Use the policy context and available tools to answer the question.
Return the required four-column JSON only.
"""
        }
    ]

    for turn in range(max_turns):

        response = call_llm(
            messages,
            tools=openai_tools,
            temperature=0.0
        )

        assistant_message = response.choices[0].message

        # If the model does not request any tools, return its answer
        if not assistant_message.tool_calls:
            return assistant_message.content or ""

        # Add assistant tool-call message to conversation
        messages.append(
            assistant_message.model_dump(exclude_none=True)
        )

        # Execute each requested tool
        for tool_call in assistant_message.tool_calls:

            tool_name = tool_call.function.name

            try:
                arguments = json.loads(
                    tool_call.function.arguments
                )

                if tool_name in TOOL_FUNCTIONS:
                    tool_result = TOOL_FUNCTIONS[tool_name](**arguments)
                else:
                    tool_result = json.dumps({
                        "error": f"Unknown tool: {tool_name}"
                    })

            except Exception as e:
                tool_result = json.dumps({
                    "error": str(e)
                })

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(tool_result)
            })

    # Ask for final JSON after tool execution limit
    messages.append({
        "role": "user",
        "content": """
Now provide the final answer using the required four-column JSON format.
Do not call any more tools. Return JSON only.
"""
    })

    final_response = call_llm(
        messages,
        temperature=0.0
    )

    return final_response.choices[0].message.content or ""


# Response parsing
def parse_response(text):
    """Extract the 4-column JSON from the agent's response."""

    patterns = [
        r'```json\s*({.*?})\s*```',
        r'({\s*"answerable"\s*:.*?})',
    ]

    for pat in patterns:
        m = re.search(pat, text, re.DOTALL)

        if m:
            try:
                data = json.loads(m.group(1))

                result = {
                    "answerable": int(data.get("answerable", 0)),
                    "lookup_value": float(data.get("lookup_value", -1)),
                    "rule_value": float(data.get("rule_value", -1)),
                    "final_answer": float(data.get("final_answer", -1)),
                }

                if result["answerable"] == 0:
                    result["lookup_value"] = -1
                    result["rule_value"] = -1
                    result["final_answer"] = -1

                return result

            except (json.JSONDecodeError, ValueError, TypeError):
                continue

    return {
        "answerable": 0,
        "lookup_value": -1,
        "rule_value": -1,
        "final_answer": -1
    }


print("Agent pipeline ready.")


Agent pipeline ready.


## Cell 10 -- Test Your Agent

Test with training queries to validate your pipeline before running on test data.

In [14]:

# CELL 10: Test Your Agent

# Pick a few training queries to test
sample_queries = train_df.sample(3, random_state=42)

for _, row in sample_queries.iterrows():

    qid = row["query_id"]
    query = row["query"]

    print(f"\n{'='*60}")
    print(f"QUERY ID: {qid}")
    print(f"QUESTION: {query}")

    print(
        f"EXPECTED: a={row['answerable']}, "
        f"l={row['lookup_value']}, "
        f"r={row['rule_value']}, "
        f"f={row['final_answer']}"
    )

    # Retrieve relevant policy documents
    policies = retrieve_policies(query)
    policy_text = "\n".join([p["text"] for p in policies])

    # Run the agent
    response = run_agent(query, policy_text)

    # Parse the agent response
    answer = parse_response(response)

    print(
        f"PREDICTED: a={answer['answerable']}, "
        f"l={answer['lookup_value']}, "
        f"r={answer['rule_value']}, "
        f"f={answer['final_answer']}"
    )

    print("RAW RESPONSE:", response)

print("\nTest complete.")



QUERY ID: Q0215
QUESTION: What is the CGPA of Soham Gupta in semester 1?
EXPECTED: a=0, l=-1.0, r=-1.0, f=-1.0
PREDICTED: a=0, l=-1, r=-1, f=-1
RAW RESPONSE: {
  "answerable": 0,
  "lookup_value": -1,
  "rule_value": -1,
  "final_answer": -1
}

QUERY ID: Q0127
QUESTION: How many students are in BATCH-2024-B?
EXPECTED: a=1, l=200.0, r=0.0, f=200.0
PREDICTED: a=0, l=-1, r=-1, f=-1
RAW RESPONSE: {
  "answerable": 0,
  "lookup_value": -1,
  "rule_value": -1,
  "final_answer": -1
}

QUERY ID: Q0175
QUESTION: How many additional lecture sessions (present) does Gagandeep Joshi (ID: STU0545) need to reach 75% weighted attendance?
EXPECTED: a=1, l=73.42, r=75.0, f=4.0
PREDICTED: a=0, l=-1, r=-1, f=-1
RAW RESPONSE: {
  "answerable": 0,
  "lookup_value": -1,
  "rule_value": -1,
  "final_answer": -1
}

Test complete.


## Cell 11 -- Generate Submission

Process all 50 test queries and write `submission.csv`.

> **This cell MUST be executed** for your notebook to produce a valid Kaggle submission.
>
> Do not modify the output format.

In [15]:
results = []
errors = []

print(f"Processing {len(test_df)} test queries...\n")

for i, row in test_df.iterrows():
    qid = row["query_id"]
    query = row["query"]

    print(f"[{i+1}/{len(test_df)}] {qid}...", end=" ")

    try:
        # Step 1: Retrieve relevant policies
        policies = retrieve_policies(query, top_k=3)
        policy_text = "\n---\n".join([p["text"] for p in policies])

        # Step 2: Run agent
        response = run_agent(query, policy_text)

        # Step 3: Parse response
        answer = parse_response(response)
        answer["query_id"] = qid
        results.append(answer)

        print(f"a={answer['answerable']}, f={answer['final_answer']}")

        # Save progress every 10 queries so a crash does not lose work
        if (i + 1) % 10 == 0:
            pd.DataFrame(results).to_csv("submission_partial.csv", index=False)

    except Exception as e:
        print(f"ERROR: {e}")
        errors.append((qid, str(e)))
        results.append({
            "query_id": qid,
            "answerable": 0,
            "lookup_value": -1,
            "rule_value": -1,
            "final_answer": -1,
        })

print(f"\nDone. {len(results)} results, {len(errors)} errors.")

# Write submission
submission = pd.DataFrame(results)[["query_id", "answerable", "lookup_value", "rule_value", "final_answer"]]
submission.to_csv("submission.csv", index=False)
print(f"\nsubmission.csv written ({len(submission)} rows).")

Processing 50 test queries...

[1/50] Q0251... a=1, f=4.0
[2/50] Q0253... a=0, f=-1
[3/50] Q0254... a=1, f=65.31
[4/50] Q0255... a=1, f=87.88
[5/50] Q0257... a=0, f=-1
[6/50] Q0258... a=1, f=100.0
[7/50] Q0259... a=1, f=2.0
[8/50] Q0260... a=1, f=1.0
[9/50] Q0262... a=1, f=4.0
[10/50] Q0264... a=1, f=1.0
[11/50] Q0267... a=0, f=-1
[12/50] Q0268... a=1, f=73.8
[13/50] Q0270... a=0, f=-1
[14/50] Q0272... a=1, f=2.14
[15/50] Q0273... a=1, f=0.0
[16/50] Q0275... a=1, f=0.0
[17/50] Q0278... a=1, f=0.0
[18/50] Q0279... a=0, f=-1
[19/50] Q0281... a=0, f=-1
[20/50] Q0282... a=0, f=-1
[21/50] Q0284... a=0, f=-1
[22/50] Q0287... a=0, f=-1
[23/50] Q0293... a=0, f=-1
[24/50] Q0294...   [retry 1/6] Error code: 400 - {'error': {'message': "Tool call validation failed: tool call validation failed: attempted to call too -- waiting 1s...
  [retry 2/6] Error code: 400 - {'error': {'message': "Tool call validation failed: tool call validation failed: attempted to call too -- waiting 2s...
a=0, f=-1
[25/5

## Cell 12 -- Final Checklist

> Run this cell to validate your submission before uploading.

In [16]:
print("Final Submission Check")
print("=" * 50)

if os.path.exists("submission.csv"):
    sub = pd.read_csv("submission.csv")

    checks = {
        "File exists": True,
        "Has 50 rows": len(sub) == 50,
        "Has required columns": all(c in sub.columns for c in ["query_id", "answerable", "lookup_value", "rule_value", "final_answer"]),
        "answerable is 0 or 1": sub["answerable"].isin([0, 1]).all(),
        "No NaN values": not sub.isnull().any().any(),
    }

    for check, passed in checks.items():
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {check}")

    oos_count = (sub["answerable"] == 0).sum()
    print(f"\n  Answerable: {len(sub) - oos_count}, Out-of-scope: {oos_count}")

    if all(checks.values()):
        print("\nAll checks passed. Ready to submit!")
    else:
        print("\nSome checks failed. Fix issues and regenerate.")
else:
    print("submission.csv not found. Run Cell 11 first.")

print("=" * 50)
print("Upload submission.csv to Kaggle to complete your submission.")

Final Submission Check
  [PASS] File exists
  [PASS] Has 50 rows
  [PASS] Has required columns
  [PASS] answerable is 0 or 1
  [PASS] No NaN values

  Answerable: 21, Out-of-scope: 29

All checks passed. Ready to submit!
Upload submission.csv to Kaggle to complete your submission.


In [17]:
import os

print("Current folder:", os.getcwd())
print("CSV files:", [f for f in os.listdir() if f.endswith(".csv")])
print("Submission exists:", os.path.exists("submission.csv"))

Current folder: /kaggle/working
CSV files: ['submission_partial.csv', 'submission.csv']
Submission exists: True


In [20]:
from IPython.display import display, HTML
import base64

with open("/kaggle/working/submission.csv", "rb") as f:
    data = base64.b64encode(f.read()).decode()

display(HTML(
    f'<a download="submission.csv" href="data:text/csv;base64,{data}">'
    'Click here to Download submission.csv</a>'
))